# Homework: Data Exploration

## Name: <span style="color:blue"> *Abby Graber* </span>

## Utils

In [ ]:
from typing import List, Dict, Tuple
import os
import gc
import traceback
import warnings
from pdb import set_trace

# Default seed
seed = 0

In [ ]:
class TodoCheckFailed(Exception):
    pass

def todo_check(asserts, mute=False, **kwargs):
    locals().update(kwargs)
    failed_err = "You passed {}/{} and FAILED the following code checks:\n{}"
    failed = ""
    n_failed = 0
    for check, (condi, err) in enumerate(asserts):
        exc_failed = False
        if isinstance(condi, str):
            try:
                passed = eval(condi)
            except Exception:
                exc_failed = True
                n_failed += 1
                failed += f"\nCheck [{check+1}]: Failed to execute check [{check+1}] due to the following error...\n{traceback.format_exc()}"
        elif isinstance(condi, bool):
            passed = condi
        else:
            raise ValueError("asserts must be a list of strings or bools")

        if not exc_failed and not passed:
            n_failed += 1
            failed += f"\nCheck [{check+1}]: Failed\n\tTip: {err}\n"

    if len(failed) != 0:
        passed = len(asserts) - n_failed
        err = failed_err.format(passed, len(asserts), failed)
        raise TodoCheckFailed(err.format(failed))
    if not mute: print("Your code PASSED all the code checks!")


# Data Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
print(f"The current path for your notebook is:\n {os.getcwd()}\n")
print(f"Your notebook is currently in the following directory:\n {os.path.basename(os.getcwd())}")

In [ ]:
df_22 = None
df_13 = None
df_05 = None

# Load datasets
df_22 = pd.read_csv('22zpallagi.csv')
df_13 = pd.read_csv('13zpallagi.csv') 
df_05 = pd.read_csv('zipcode05.csv') 

#### Table Display

In [ ]:
# 2022 Data
display(df_22)

In [ ]:
# 2010 Data
display(df_13)

In [ ]:
#1998 Data
display(df_05)

In [ ]:
# Get and display column names (features)
feature_names_22 = None
feature_names_13 = None
feature_names_05 = None

feature_names_22 = df_22.columns
feature_names_13 = df_13.columns
feature_names_05 = df_05.columns

print(f'The feature names for 2022 are:\n{feature_names_22.values}\n')
print(f'The feature names for 2013 are:\n{feature_names_13.values}\n')
print(f'The feature names for 2005 are:\n{feature_names_05.values}\n')

### <span style="color:white"> Features to Include </span>
   `zipcode`  

   `state`  

   `N1` (Number of returns)  

   `A00100` (Total AGI)  

   `agi_stub` (Income bracket)  

We need to add a `year` column in order to differentiate each dataset when comparing them.

Consolodate datasets for data visualization and add a `year` column

In [ ]:
# Define the columns we need
cols = ['STATE', 'zipcode', 'N1', 'A00100', 'agi_stub']

# Reload each dataframe with only the specified columns
df_22 = pd.read_csv('22zpallagi.csv', usecols=cols, dtype={'zipcode': str})
df_13 = pd.read_csv('13zpallagi.csv', usecols=cols, dtype={'zipcode': str}) 
df_05 = pd.read_csv('zipcode05.csv', usecols=cols, dtype={'zipcode': str}) 

# Add year to each dataframe
df_22['YEAR'] = 2022
df_13['YEAR'] = 2013
df_05['YEAR'] = 2005

# Get and display new column names (features)
feature_names_22 = None
feature_names_13 = None
feature_names_05 = None

feature_names_22 = df_22.columns
feature_names_13 = df_13.columns
feature_names_05 = df_05.columns

print(f'The feature names for 2022 are:\n{feature_names_22.values}\n')
print(f'The feature names for 2013 are:\n{feature_names_13.values}\n')
print(f'The feature names for 2005 are:\n{feature_names_05.values}\n')

# Master Dataframe

Combine all datasets into one master dataframe

In [ ]:
master_df = pd.concat([df_05, df_13, df_22],ignore_index=True)

print(master_df.head())
print(master_df.tail())

# Data Visualization and Exploration

## Exploring


In [ ]:
d22_shape = None
d13_shape = None
d05_shape = None
master_shape = None

d22_shape = df_22.shape
d13_shape = df_13.shape
d05_shape = df_05.shape
master_shape = master_df.shape

print(f'The 2022 dataset shape is: {d22_shape}')
print(f'The 2013 dataset shape is: {d13_shape}')
print(f'The 2005 dataset shape is: {d05_shape}')
print(f'The master dataset shape is: {master_shape}')

In [ ]:
# Display summary of the data
print(df_22.info())  
print(df_13.info())  
print(df_05.info())
print(master_df.info())  

## Visualization

### Total AGI % change per ZIP

Shows long-term vs recent trends (2005 → 2013 → 2022).  
Detects areas that were historically growing but are now declining (i.e. emerging risk zones).  

Chart Captures -  
Total welth flowing in or out of ZIP  
Economic size expansion or contraction  
Exposure risk for a bank lending in that region 

In [ ]:
# Aggregragte total AGI per ZIP per year
zip_year_agi = (
    master_df
    .groupby(['zipcode', 'YEAR'])['A00100']
    .sum()
    .reset_index()
)

# Pivot years to columns
zip_pivot = zip_year_agi.pivot(
    index='zipcode',
    columns='YEAR',
    values='A00100'
)

zip_pivot = zip_pivot.dropna()

# Compute all three period changes
zip_pivot['pct_05_13'] = ((zip_pivot[2013] - zip_pivot[2005]) / zip_pivot[2005]) * 100
zip_pivot['pct_13_22'] = ((zip_pivot[2022] - zip_pivot[2013]) / zip_pivot[2013]) * 100
zip_pivot['pct_05_22'] = ((zip_pivot[2022] - zip_pivot[2005]) / zip_pivot[2005]) * 100

# Select extreme zipcodes
top_n = 10

top_growth = zip_pivot.sort_values('pct_05_22', ascending=False).head(top_n)
top_decline = zip_pivot.sort_values('pct_05_22').head(top_n)

plot_df = pd.concat([top_decline, top_growth])
plot_df = plot_df.sort_values('pct_05_22')

# Plotting function
def plot_diverging(ax, data, column, title):
    colors = ['red' if x < 0 else 'green' for x in data[column]]
    
    ax.barh(
        data.index,
        data[column],
        color=colors
    )
    
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('Percent Change')
    ax.set_xlim(-100, 200)  # adjust if needed

# Create 3 diverging charts
fig, axes = plt.subplots(3, 1, figsize=(10, 18))

plot_diverging(axes[0], plot_df, 'pct_05_13', 'AGI % Change (2005 → 2013)')
plot_diverging(axes[1], plot_df, 'pct_13_22', 'AGI % Change (2013 → 2022)')
plot_diverging(axes[2], plot_df, 'pct_05_22', 'AGI % Change (2005 → 2022)')

plt.tight_layout()
plt.show()

In [ ]:
# Average AGI
master_df['avg_agi'] = master_df['A00100'] / master_df['N1']

# Pivot for easier comparison
pivot_df = master_df.pivot_table(index='zipcode', columns='YEAR', values=['A00100', 'N1', 'avg_agi'])

pivot_df.columns = [f"{col[0]}_{col[1]}" for col in pivot_df.columns]

pivot_feature_names = None
pivot_feature_names = pivot_df.columns
print(f'The feature names for the pivot dataframe are:\n{pivot_feature_names.values}\n')

### Average AGI per tax return % change

Highlights the ZIP codes with the largest positive and negative changes.  
Quickly identifies “red flag” areas where lending risk is highest.  
Centered at 0 → easy to see shrinking vs booming areas.  

Chart captures:  
Change in income per household  
Income quality shifts  
Gentrification vs middle-class erosion  

In [ ]:
import seaborn as sns

# Compute AGI per return & % change
pivot_df["avg_agi_2005"] = pivot_df["A00100_2005"] / pivot_df["N1_2005"]
pivot_df["avg_agi_2022"] = pivot_df["A00100_2022"] / pivot_df["N1_2022"]

pivot_df["avg_agi_pct_change"] = ((pivot_df["avg_agi_2022"] - pivot_df["avg_agi_2005"]) / pivot_df["avg_agi_2005"]) * 100

# Select Top/Bottom  ZIP codes
N = 15  # number of ZIPs to show for each side

top_zips = pivot_df.nlargest(N, "avg_agi_pct_change")
bottom_zips = pivot_df.nsmallest(N, "avg_agi_pct_change")

top_bottom = pd.concat([bottom_zips, top_zips])

# Sort for plotting
top_bottom = top_bottom.sort_values("avg_agi_pct_change")

plt.figure(figsize=(10, 8))
sns.set_style("whitegrid")

barplot = sns.barplot(
    data=top_bottom,
    y="zipcode",
    x="avg_agi_pct_change",
    palette=["red" if x < 0 else "blue" for x in top_bottom["avg_agi_pct_change"]]
)

plt.axvline(0, color="black", linewidth=1)  # center line at 0
plt.title("Top/Bottom ZIP Codes by % Change in AGI per Return (2005 → 2022)", fontsize=14)
plt.xlabel("% Change in AGI per Return")
plt.ylabel("ZIP Code")
plt.tight_layout()
plt.show()


### Change in average income per tax return at the ZIP-code level (2005 → 2022)

Compares absolute household income per filer in 2005 vs 2022.  
Points above diagonal = improving households; below = weakening households.  

Chart Captures -
Income growth or decline per filer  
Above diagonal (y > x) → income per filer increased  
Below diagonal (y < x) → income per filer decreased  

Each dot represents a ZIP

In [ ]:
# Compute AGI per return
pivot_df["avg_agi_2005"] = pivot_df["A00100_2005"] / pivot_df["N1_2005"]
pivot_df["avg_agi_2022"] = pivot_df["A00100_2022"] / pivot_df["N1_2022"]

# Plot Scatter
plt.figure(figsize=(10, 8))
sns.set_style("whitegrid")

# Scatter points
sns.scatterplot(
    data=pivot_df,
    x="avg_agi_2005",
    y="avg_agi_2022",
    alpha=0.6,
    s=50,
    color="dodgerblue"
)

# Diagonal line y = x
max_val = max(pivot_df["avg_agi_2005"].max(), pivot_df["avg_agi_2022"].max())
plt.plot([0, max_val], [0, max_val], color="red", linestyle="--", linewidth=1)

plt.xlim(0, 4000) 
plt.ylim(0, 5000) 

plt.title("Historical vs Current AGI per Return by ZIP (2005 → 2022)", fontsize=14)
plt.xlabel("Historical AGI per Return (2005)")
plt.ylabel("Current AGI per Return (2022)")

plt.tight_layout()
plt.show()

### Average AGI per return & Percent change per ZIP & Histogram + KDE

Shows the overall pattern across all ZIP codes, not just extremes.  
Captures the proportion of ZIPs experiencing growth vs decline, and the magnitude of changes.  

Focuses on income per filer  
Shows relative growth or decline for each ZIP.  
X-axis = % change in avg AGI per return  
Y-axis = number of ZIP codes  
KDE line shows a smooth estimate of the distribution.  

Chart Captures -  
Overall trends in household income across ZIP codes  
Proportion of ZIPs growing vs shrinking  
Magnitude of change  
Income polarization or stability

In [ ]:
# Compute AGI per return & % change
pivot_df["avg_agi_2005"] = pivot_df["A00100_2005"] / pivot_df["N1_2005"]
pivot_df["avg_agi_2022"] = pivot_df["A00100_2022"] / pivot_df["N1_2022"]

pivot_df["avg_agi_pct_change"] = ((pivot_df["avg_agi_2022"] - pivot_df["avg_agi_2005"]) / pivot_df["avg_agi_2005"]) * 100

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

sns.histplot(
    pivot_df["avg_agi_pct_change"],
    bins=50,
    kde=True,
    color="dodgerblue",
    edgecolor="black"
)

plt.axvline(0, color="red", linestyle="--", linewidth=1)  # zero-change line

plt.title("Distribution of % Change in AGI per Return (2005 → 2022)", fontsize=14)
plt.xlabel("% Change in AGI per Return")
plt.ylabel("Number of ZIP Codes")

plt.tight_layout()
plt.show()

### Line Chart
Rising line: Increasing household income → lower lending risk  
Falling line: Shrinking household income → higher lending risk  
Line that peaks then drops: Emerging risk zone — previously strong, now deteriorating  

In [ ]:
# Compute avg AGI per filer
avg_n_returns = master_df.groupby('zipcode')['N1'].sum()
zip_pivot['avg_agi_2005'] = zip_pivot[2005] / avg_n_returns
zip_pivot['avg_agi_2013'] = zip_pivot[2013] / avg_n_returns
zip_pivot['avg_agi_2022'] = zip_pivot[2022] / avg_n_returns

# Select top shrinking / growing ZIPs
zip_pivot['pct_change'] = ((zip_pivot['avg_agi_2022'] - zip_pivot['avg_agi_2005']) / zip_pivot['avg_agi_2005']) * 100

top_growth = zip_pivot.nlargest(5, 'pct_change')
top_decline = zip_pivot.nsmallest(5, 'pct_change')

# Combine for plotting
plot_df = pd.concat([top_decline, top_growth])

# Prepare data for line plotting (rows = years, columns = ZIPs)
plot_df = plot_df[['avg_agi_2005','avg_agi_2013','avg_agi_2022']].transpose()
plot_df.index = [2005, 2013, 2022]

# Assign unique colors to each ZIP
zips = plot_df.columns
n_zips = len(zips)
palette = sns.color_palette("tab10", n_colors=n_zips)  # one color per ZIP

# Plot line chart
plt.figure(figsize=(12, 6))

for i, zip_code in enumerate(zips):
    plt.plot(
        plot_df.index,
        plot_df[zip_code],
        marker='o',
        color=palette[i],      # unique color per ZIP
        label=zip_code,
        linewidth=2,
        alpha=0.8
    )

plt.title("Average AGI per Return by ZIP (2005 → 2013 → 2022)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Average AGI per Return")
plt.xticks([2005, 2013, 2022])
plt.grid(True)
plt.legend(title="ZIP Code", bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.show()